**Source of the materials**: Biopython Tutorial and Cookbook (adapted)  
**Lab Reference**: [Exploring interactive phylogenies with Auspice – Neherlab](https://neherlab.org/201901_krisp_auspice.html)

# Lab 03: Phylogenetics with Bio.Phylo

In this lab we explore phylogenetic tree construction, manipulation, and visualization using **Biopython's `Bio.Phylo` module**.  
We also connect these concepts to interactive phylogenomic visualization using **Nextstrain / Auspice**, as described in the Neherlab tutorial.

### Learning Objectives
- Read and write phylogenetic tree files (Newick, PhyloXML)
- Traverse, search, and manipulate tree objects
- Visualize trees with `Bio.Phylo` and `matplotlib`
- Understand what Nextstrain/Auspice adds on top of basic phylogenetics

## 0. Setup – Install Dependencies

Run this cell once in Colab to install all required packages.

In [ ]:
# Uncomment and run in Colab
# !pip install biopython --quiet
import sys
print(f"Python {sys.version}")
import Bio
print(f"Biopython {Bio.__version__}")

## 1. Demo: What Is in a Tree?

Let's start by opening an example Newick file and inspecting the tree object.

In [ ]:
import copy
from io import StringIO

from Bio import Phylo
from Bio.Phylo.PhyloXML import Phylogeny

%matplotlib inline

tree = Phylo.read("data/simple.dnd", "newick")

Printing the tree object as a string gives us a look at the entire object hierarchy.

In [ ]:
print(tree)

The `Phylo.draw_ascii()` function gives a quick text-based visualization.

In [ ]:
Phylo.draw_ascii(tree)

## 2. Traversal Functions

The `Tree` object has methods to traverse all nodes, get tips (terminals), and get internal nodes.

In [ ]:
# Get all terminal (leaf) nodes
print("Terminals (leaves):")
for tip in tree.get_terminals():
    print(f"  {tip.name}  (branch length: {tip.branch_length})")

In [ ]:
# Get all internal nodes
print("Internal nodes:")
for node in tree.get_nonterminals():
    print(f"  confidence={node.confidence}  branch_length={node.branch_length}")

In [ ]:
# Count total leaves
print(f"Total leaves: {tree.count_terminals()}")

# Total branch length (sum of all branches)
print(f"Total branch length: {tree.total_branch_length():.4f}")

## 3. Searching the Tree

`find_elements()` is a flexible search tool. We can search by name, branch length, or any attribute.

In [ ]:
# Find a specific leaf by name
target = tree.find_any("A")
print(f"Found: {target}")
print(f"  Branch length: {target.branch_length}")

In [ ]:
# Find the common ancestor of two taxa
mrca = tree.common_ancestor("E", "F")
print(f"Common ancestor of E and F: {mrca}")
print(f"  Branch length: {mrca.branch_length}")

In [ ]:
# Distance between two leaves
dist = tree.distance("A", "B")
print(f"Distance between A and B: {dist:.4f}")

dist_ef = tree.distance("E", "F")
print(f"Distance between E and F: {dist_ef:.4f}")

## 4. Modifying the Tree

We can root the tree, prune leaves, and collapse nodes.

In [ ]:
# Make a copy so we don't lose the original
tree_mod = copy.deepcopy(tree)

# Root the tree at the midpoint
tree_mod.root_at_midpoint()
print("Tree rooted at midpoint:")
Phylo.draw_ascii(tree_mod)

In [ ]:
# Prune a leaf
tree_pruned = copy.deepcopy(tree)
tree_pruned.prune("D")
print("Tree after pruning leaf D:")
Phylo.draw_ascii(tree_pruned)

## 5. Coloring and Annotating the Tree

Biopython lets us assign colours and widths to branches. These are supported in the PhyloXML format.

In [ ]:
# Convert to PhyloXML to enable color/width attributes
tree_color = copy.deepcopy(tree)
tree_color = tree_color.as_phyloxml()
tree_color = Phylogeny.from_tree(tree_color)

# Color the clade containing E and F red
ef_ancestor = tree_color.common_ancestor("E", "F")
ef_ancestor.color = "red"

# Color the clade containing A and B blue
ab_ancestor = tree_color.common_ancestor("A", "B")
ab_ancestor.color = "blue"

print("Colored tree (color visible in Phylo.draw):")
Phylo.draw_ascii(tree_color)

## 6. Visualizing with Matplotlib

`Phylo.draw()` renders a publication-style figure.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original tree
Phylo.draw(tree, axes=axes[0], do_show=False)
axes[0].set_title("Original Tree")

# Colored tree
Phylo.draw(tree_color, axes=axes[1], do_show=False)
axes[1].set_title("Colored Tree (E-F: red, A-B: blue)")

plt.tight_layout()
plt.savefig("outputs/tree_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to outputs/tree_visualization.png")

## 7. I/O Functions

Like `SeqIO` and `AlignIO`, Phylo handles I/O through `parse`, `read`, `write`, and `convert`.

In [ ]:
# Write the colored tree to PhyloXML format
Phylo.write(tree_color, "outputs/colored_tree.xml", "phyloxml")
print("Written to outputs/colored_tree.xml")

# Write same tree in Newick format
Phylo.write(tree, "outputs/tree_out.nwk", "newick")
print("Written to outputs/tree_out.nwk")

In [ ]:
# Read it back and verify
tree_back = Phylo.read("outputs/colored_tree.xml", "phyloxml")
print("Read back from PhyloXML:")
print(tree_back)
print(f"\nLeaves: {[tip.name for tip in tree_back.get_terminals()]}")

In [ ]:
# Convert directly between formats
Phylo.convert("data/simple.dnd", "newick", "outputs/converted.xml", "phyloxml")
print("Converted Newick -> PhyloXML successfully")

## 8. Building a Tree from a Distance Matrix (UPGMA)

We can construct a tree from pairwise distances using the UPGMA or Neighbor-Joining algorithms.

In [ ]:
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor

# Define taxa and a lower-triangular distance matrix
names = ["Seq1", "Seq2", "Seq3", "Seq4", "Seq5"]
matrix = [
    [0],
    [5, 0],
    [9, 10, 0],
    [9, 10, 8, 0],
    [8, 9, 7, 3, 0],
]

dm = DistanceMatrix(names, matrix)
print(dm)

In [ ]:
constructor = DistanceTreeConstructor()

# UPGMA tree
upgma_tree = constructor.upgma(dm)
print("UPGMA Tree:")
Phylo.draw_ascii(upgma_tree)

# Neighbor-Joining tree
nj_tree = constructor.nj(dm)
print("\nNeighbor-Joining Tree:")
Phylo.draw_ascii(nj_tree)

In [ ]:
# Plot both side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

Phylo.draw(upgma_tree, axes=axes[0], do_show=False)
axes[0].set_title("UPGMA Tree")

Phylo.draw(nj_tree, axes=axes[1], do_show=False)
axes[1].set_title("Neighbor-Joining Tree")

plt.tight_layout()
plt.savefig("outputs/upgma_vs_nj.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Building a Tree from a Multiple Sequence Alignment

A more realistic workflow: align sequences, compute pairwise distances, then build a tree.

In [ ]:
from Bio import AlignIO
from Bio.Phylo.TreeConstruction import DistanceCalculator

# Create a small mock alignment (FASTA-like)
aln_str = """\
>Taxon_A
ATGCTAGCTAGCTAGCTAGCTA
>Taxon_B
ATGCTAGCTAGCTAGCTAGCTG
>Taxon_C
ATGCTAGCAAGCTAGCTAGCTA
>Taxon_D
ATGCTAGCAAGCAAGCTAGCTA
>Taxon_E
ATGCAAGCAAGCAAGCTAGCTA
"""

from Bio import SeqIO
from Bio.Align import MultipleSeqAlignment
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

records = list(SeqIO.parse(StringIO(aln_str), "fasta"))
aln = MultipleSeqAlignment(records)
print(f"Alignment: {len(aln)} sequences × {aln.get_alignment_length()} positions")
print(aln)

In [ ]:
# Calculate pairwise distances from the alignment
calculator = DistanceCalculator("identity")
dm_aln = calculator.get_distance(aln)
print(dm_aln)

# Build NJ tree
constructor2 = DistanceTreeConstructor(calculator, "nj")
nj_aln_tree = constructor2.build_tree(aln)
print("\nNJ Tree from alignment:")
Phylo.draw_ascii(nj_aln_tree)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
Phylo.draw(nj_aln_tree, axes=ax, do_show=False)
ax.set_title("NJ Tree from Sequence Alignment")
plt.tight_layout()
plt.savefig("outputs/nj_from_alignment.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Connection to Nextstrain / Auspice

The [Neherlab Auspice tutorial](https://neherlab.org/201901_krisp_auspice.html) demonstrates how **Nextstrain** extends classical phylogenetics with:

| Feature | Bio.Phylo | Nextstrain / Auspice |
|---|---|---|
| Tree construction | ✅ Newick / NJ / UPGMA | ✅ IQ-TREE / FastTree + temporal signal |
| Visualization | Static matplotlib | Interactive browser-based |
| Geographic mapping | ❌ | ✅ Animated map |
| Ancestral state reconstruction | ❌ | ✅ (location, genotype) |
| Diversity panel (entropy) | ❌ | ✅ Per-site entropy across genome |
| Time-resolved tree | Manual | ✅ Molecular clock model via TreeTime |

### Key Auspice concepts
- **Time-resolved tree**: x-axis is calendar time; branch lengths encode elapsed time (not raw substitutions)
- **Clock view**: scatter of divergence vs. sampling date; slope = evolutionary rate (subs/site/year)
- **Colour-by**: phylogeny recoloured by country, genotype, author, date — equivalent to querying tip metadata
- **Diversity panel**: per-position entropy or event counts across the genome; clicking a site recolours the tree by genotype at that position
- **Geographic spread animation**: ancestral location reconstruction shown over time on a world map

### Try it yourself
Open the [Zika tutorial on Nextstrain](https://nextstrain.org/community/neherlab/krisp/zika-tutorial) and explore:
1. Mouse over branches — how many descendants does the large mid-tree branch have?
2. Switch Color-by to **region** and then **date**.
3. Switch to **Clock** layout — what is the approximate evolutionary rate (slope)?
4. In the Diversity panel, click the highest-entropy AA site. Which gene and position is it in? What amino acids are present in the tree?
5. Type `3501,3972,6966` in the NT genotype colour-by box. Describe what you observe.

## 11. Summary

In this lab we covered:

1. **Reading and printing** Newick and PhyloXML trees with `Bio.Phylo`
2. **Traversal**: `get_terminals()`, `get_nonterminals()`, `count_terminals()`
3. **Searching**: `find_any()`, `common_ancestor()`, `distance()`
4. **Modification**: `root_at_midpoint()`, `prune()`
5. **Colour annotations** in PhyloXML
6. **Visualization** with `Phylo.draw()` and matplotlib
7. **I/O**: `Phylo.write()`, `Phylo.convert()`
8. **Tree construction** from distance matrices (UPGMA, NJ) and sequence alignments
9. **Connection** to Nextstrain/Auspice for interactive, time-resolved, geographically-annotated phylogenies